### Notebook 01 — Data Extraction Pipeline
#### Market Pulse Analytics

**Business Purpose:**
This notebook builds the data ingestion layer of the analytics pipeline.
It extracts 5 years of daily equity data for 12 stocks across 4 sectors
and saves both raw and combined datasets for downstream processing.

**Source:** Yahoo Finance (yfinance library)
**Period:** January 2020 to present (~1,260 trading days per ticker)

In [8]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import time
import logging
import warnings
from datetime import datetime

import yfinance as yf
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

from config.settings import(
    TICKERS,
    SECTOR_MAP,
    COMPANY_NAMES,
    SUB_SECTOR_MAP,
    START_DATE,
    END_DATE,
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    LOG_PATH,
    API_CALL_DELAY_SECONDS,
)

# Confirm settings loaded correctly
print("=" * 55)
print("MARKET PULSE ANALYTICS — Data Extraction")
print("=" * 55)
print(f"Tickers    : {TICKERS}")
print(f"Start Date : {START_DATE}")
print(f"End Date   : {END_DATE}")
print(f"Ticker count: {len(TICKERS)}")

MARKET PULSE ANALYTICS — Data Extraction
Tickers    : ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'GS', 'BAC', 'JNJ', 'PFE', 'XOM', 'CVX', 'AMZN', 'WMT']
Start Date : 2020-01-01
End Date   : 2026-05-24
Ticker count: 12


In [9]:
# CELL 3 — Configure Logging
os.makedirs(f"../{RAW_DATA_PATH}",       exist_ok=True)
os.makedirs(f"../{PROCESSED_DATA_PATH}", exist_ok=True)
os.makedirs("../logs",                   exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(f"../{LOG_PATH}", mode="a"),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("extraction")

logger.info("=" * 55)
logger.info("Data Extraction Started")
logger.info(f"Run time  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info(f"Tickers   : {len(TICKERS)}")
logger.info(f"Date range: {START_DATE} to {END_DATE}")
logger.info("=" * 55)

2026-05-24 14:12:40 | INFO | =======================================================
2026-05-24 14:12:40 | INFO | Data Extraction Started
2026-05-24 14:12:40 | INFO | Run time  : 2026-05-24 14:12:40
2026-05-24 14:12:40 | INFO | Tickers   : 12
2026-05-24 14:12:40 | INFO | Date range: 2020-01-01 to 2026-05-24
2026-05-24 14:12:40 | INFO | =======================================================


In [16]:
# CELL 4 — Function: Fetch One Ticker

def fetch_stock_data(ticker, start_date, end_date):
    """
    xDownloads daily OHLCV data for one ticker using yfinance.
    """
    try:
        logger.info(f'Fetching: {ticker}')
        # auto_adjust=True adjusts for stock splits and dividends
        # This is critical for accurate long-term return calculations
        raw = yf.download(
            tickers = ticker,
            start = start_date,
            end = end_date,
            auto_adjust = True,
            progress = False,
        )
        if raw is None or raw.empty:
            logger.warning(f'No data returned for {ticker}')
        
        raw = raw.reset_index()
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = [col[0] if col[1] == "" else col[0]
                           for col in raw.columns]
        # Standardize column names to lowercase snake_case
        raw.columns = [str(c).lower().strip().replace(" ", "_")
                       for c in raw.columns]
        
        # Rename price columns explicitly for clarity
        col_map = {}
        for c in raw.columns:
            if c in ["open"]:   col_map[c] = "open"
            if c in ["high"]:   col_map[c] = "high"
            if c in ["low"]:    col_map[c] = "low"
            if c in ["close"]:  col_map[c] = "close"
            if c in ["volume"]: col_map[c] = "volume"
        raw = raw.rename(columns=col_map)

        # Add identifier columns
        raw["ticker"]  = ticker
        raw["sector"]  = SECTOR_MAP.get(ticker, "Unknown")
        raw["company"] = COMPANY_NAMES.get(ticker, ticker)

        raw['date'] = pd.to_datetime(raw['date'])

        # Drop rows where close price is missing
        raw = raw.dropna(subset = ['close'])

        # Volume to integer
        if 'volume' in raw.columns:
            raw['volume'] = raw['volume'].fillna(0).astype(int)

        # Keep only the columns we need — in a clean order
        keep_cols = ["date", "ticker", "company", "sector",
                     "open", "high", "low", "close", "volume"]
        keep_cols = [c for c in keep_cols if c in raw.columns]
        raw = raw[keep_cols]

        logger.info(
            f" [SUCCESS] {ticker}: {len(raw):,} rows | "
            f"{raw['date'].min().date()} to {raw['date'].max().date()}"
        )

        return raw
    
    except Exception as e:
        logger.error(f'[FAILED] {ticker} failed: {str(e)}')
        return None
    

# ── Test on one ticker before running all 12 ──────────────────
test_df = fetch_stock_data("AAPL", START_DATE, END_DATE)

if test_df is not None:
    print("\n--- AAPL Sample ---")
    print(test_df.head(5).to_string(index=False))
    print(f"\nShape : {test_df.shape}")
    print(f"Dtypes:\n{test_df.dtypes}") 

2026-05-24 15:18:38 | INFO | Fetching: AAPL
2026-05-24 15:18:38 | INFO |  [SUCCESS] AAPL: 1,606 rows | 2020-01-02 to 2026-05-22

--- AAPL Sample ---
      date ticker    company     sector      open      high       low     close    volume
2020-01-02   AAPL Apple Inc. Technology 71.344039 72.394070 71.091169 72.333862 135480400
2020-01-03   AAPL Apple Inc. Technology 71.563198 72.389250 71.406659 71.630630 146322800
2020-01-06   AAPL Apple Inc. Technology 70.754014 72.239942 70.503546 72.201408 118387200
2020-01-07   AAPL Apple Inc. Technology 72.211041 72.466322 71.642681 71.861839 108872000
2020-01-08   AAPL Apple Inc. Technology 71.565629 73.318885 71.565629 73.017845 132079200

Shape : (1606, 9)
Dtypes:
date       datetime64[ns]
ticker             object
company            object
sector             object
open              float64
high              float64
low               float64
close             float64
volume              int64
dtype: object


In [18]:
# CELL 5 — Fetch All Tickers

logger.info(f"\nBeginning full extraction for {len(TICKERS)} tickers...")

collected = []
failed_list = []

for idx, ticker in enumerate(TICKERS):

    df = fetch_stock_data(ticker, START_DATE, END_DATE)

    if df is not None:
        collected.append(df)

        raw_path = f'../{RAW_DATA_PATH}{ticker}_daily_raw.csv'
        df.to_csv(raw_path, index = False)
        logger.info(f' Saved: {raw_path}')
    
    else:
        failed_list.append(ticker)

    if idx < len(TICKERS) - 1:
        time.sleep(API_CALL_DELAY_SECONDS)

# ── Combine all into master dataframe ─────────────────────────

if collected:
    master_df = pd.concat(collected, ignore_index=True)
    master_df = master_df.sort_values(['ticker', 'date']).reset_index(drop = True)
else:
    raise RuntimeError('No data collected. Check internet connection and API.')


# ── Extraction summary ────────────────────────────────────────
logger.info("\n" + "=" * 55)
logger.info("EXTRACTION SUMMARY")
logger.info(f"  Successful : {len(collected)}/{len(TICKERS)} tickers")
logger.info(f"  Failed     : {failed_list if failed_list else 'None'}")
logger.info(f"  Total rows : {len(master_df):,}")
logger.info(f"  Date range : {master_df['date'].min().date()} to {master_df['date'].max().date()}")
logger.info("=" * 55)

print(f"\n✓ Master dataset created")
print(f"  Shape    : {master_df.shape}")
print(f"\nRow count per ticker:")
print(master_df.groupby("ticker")["date"].count().to_string())

2026-05-24 15:19:36 | INFO | 
Beginning full extraction for 12 tickers...
2026-05-24 15:19:36 | INFO | Fetching: AAPL


2026-05-24 15:19:36 | INFO |  [SUCCESS] AAPL: 1,606 rows | 2020-01-02 to 2026-05-22
2026-05-24 15:19:36 | INFO |  Saved: ../data/raw/AAPL_daily_raw.csv
2026-05-24 15:19:37 | INFO | Fetching: MSFT
2026-05-24 15:19:37 | INFO |  [SUCCESS] MSFT: 1,606 rows | 2020-01-02 to 2026-05-22
2026-05-24 15:19:37 | INFO |  Saved: ../data/raw/MSFT_daily_raw.csv
2026-05-24 15:19:38 | INFO | Fetching: GOOGL
2026-05-24 15:19:39 | INFO |  [SUCCESS] GOOGL: 1,606 rows | 2020-01-02 to 2026-05-22
2026-05-24 15:19:39 | INFO |  Saved: ../data/raw/GOOGL_daily_raw.csv
2026-05-24 15:19:40 | INFO | Fetching: JPM
2026-05-24 15:19:40 | INFO |  [SUCCESS] JPM: 1,606 rows | 2020-01-02 to 2026-05-22
2026-05-24 15:19:40 | INFO |  Saved: ../data/raw/JPM_daily_raw.csv
2026-05-24 15:19:41 | INFO | Fetching: GS
2026-05-24 15:19:41 | INFO |  [SUCCESS] GS: 1,606 rows | 2020-01-02 to 2026-05-22
2026-05-24 15:19:41 | INFO |  Saved: ../data/raw/GS_daily_raw.csv
2026-05-24 15:19:42 | INFO | Fetching: BAC
2026-05-24 15:19:43 | INFO 

In [20]:
# CELL 6 — Data Quality Validation

print("=" * 55)
print("DATA QUALITY VALIDATION REPORT")
print("=" * 55)

all_checks_passed = True

# Check 1: Missing values
missing = master_df.isna().sum()
missing_cols = missing[missing>0]

print(f"\n1. Missing Values:")

if missing_cols.empty:
    print("   ✓ No missing values")
else:
    print(f"   ⚠ Found missing values:\n{missing_cols}")
    all_checks_passed = False

# Check 2: Duplicate records
dupes = master_df.duplicated(subset = ['ticker','date']).sum()

print(f"\n2. Duplicate (ticker + date) pairs: {dupes}")
if dupes == 0:
    print("   ✓ No duplicates")
else:
    print(f"   ⚠ Dropping {dupes} duplicates")
    master_df = master_df.drop_duplicates(subset=["ticker", "date"])

# Check 3: Negative or zero close prices
bad_prices = master_df[master_df['close'] <= 0]

print(f"\n3. Invalid close prices (<= 0): {len(bad_prices)}")
if bad_prices.empty:
    print("   ✓ All prices valid")
else:
    print(f"   ⚠ {bad_prices[['ticker','date','close']]}")
    all_checks_passed = False

# Check 4: Sector coverage
print(f"\n4. Coverage by sector:")
sector_counts = master_df.groupby('sector')['ticker'].nunique()
print(sector_counts.to_string())

# Check 5: Date range completeness
print(f"\n5. Date range per ticker:")
date_range = master_df.groupby("ticker").agg(
    start = ("date", "min"),
    end   = ("date", "max"),
    days  = ("date", "count")
).reset_index()
print(date_range.to_string(index=False))

# Check 6: Volume sanity
zero_vol = master_df[master_df["volume"] == 0]
print(f"\n6. Zero volume days: {len(zero_vol)}")
if len(zero_vol) > 0:
    print(zero_vol[["ticker", "date", "volume"]].head(10))

# Final verdict
print(f"\n{'=' * 55}")
if all_checks_passed:
    print("✓ All validation checks passed — data is clean")
else:
    print("⚠ Some checks flagged issues — review above before continuing")
print("=" * 55)

logger.info(f"Validation complete. Total clean rows: {len(master_df):,}")

DATA QUALITY VALIDATION REPORT

1. Missing Values:
   ✓ No missing values

2. Duplicate (ticker + date) pairs: 0
   ✓ No duplicates

3. Invalid close prices (<= 0): 0
   ✓ All prices valid

4. Coverage by sector:
sector
Consumer      2
Energy        2
Financials    3
Healthcare    2
Technology    3

5. Date range per ticker:
ticker      start        end  days
  AAPL 2020-01-02 2026-05-22  1606
  AMZN 2020-01-02 2026-05-22  1606
   BAC 2020-01-02 2026-05-22  1606
   CVX 2020-01-02 2026-05-22  1606
 GOOGL 2020-01-02 2026-05-22  1606
    GS 2020-01-02 2026-05-22  1606
   JNJ 2020-01-02 2026-05-22  1606
   JPM 2020-01-02 2026-05-22  1606
  MSFT 2020-01-02 2026-05-22  1606
   PFE 2020-01-02 2026-05-22  1606
   WMT 2020-01-02 2026-05-22  1606
   XOM 2020-01-02 2026-05-22  1606

6. Zero volume days: 0

✓ All validation checks passed — data is clean
2026-05-24 15:32:58 | INFO | Validation complete. Total clean rows: 19,272


In [ ]:
# CELL 7 — Save Master Dataset
master_path = f"../{PROCESSED_DATA_PATH}master_stock_data.csv"
master_df.to_csv(master_path, index=False)

logger.info(f"Master dataset saved: {master_path}")
logger.info(f"Total records: {len(master_df):,}")

print(f"\n✓ Saved: {master_path}")
print(f"✓ {len(master_df):,} records across {master_df['ticker'].nunique()} tickers")
print(f"✓ {master_df['date'].min().date()} to {master_df['date'].max().date()}")
print("\nNotebook 01 complete. Proceed to 02_data_cleaning.ipynb")

2026-05-24 15:39:46 | INFO | Master dataset saved: ../data/processed/master_stock_data.csv
2026-05-24 15:39:46 | INFO | Total records: 19,272

✓ Saved: ../data/processed/master_stock_data.csv
✓ 19,272 records across 12 tickers
✓ 2020-01-02 to 2026-05-22

Notebook 01 complete. Proceed to 02_data_cleaning.ipynb
